# 📘 K-Nearest Neighbors (KNN) — Concepts Notebook
### Atlas Skilltech University | Instructor Reference

---

**Yeh notebook kyun hai?** Yahan hum KNN ke har topic ko chhote chhote sample data ke saath samjhenge — taaki concepts ekdum clear ho jayein real dataset pe jaane se pehle.

**Structure (Kya kya cover karenge):**
1. KNN kya hota hai? — Concept aur Theory
2. Distance Functions — Euclidean, Manhattan, Minkowski
3. KNN kyun use karein?
4. Mathematical Approach — Step by step
5. Sample Dataset ke saath EDA
6. Preprocessing — Scaling kyun zaroori hai
7. Model Training — Best K kaise chunein
8. Confusion Matrix aur Accuracy
9. Precision, Recall, F1 Score
10. KNN Regression (bonus)
11. KNN ki Limitations

> 💡 Har mushkil keyword ko code se **pehle** markdown cell mein explain kiya gaya hai.

---
## 🔧 Step 0 — Libraries Import Karo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Saari libraries successfully import ho gayi ✅')

---
## 📌 Section 1 — KNN Kya Hota Hai? (Concept aur Theory)

**Yeh keywords yaad rakho:**
- **K-Nearest Neighbors (KNN)** — ek simple machine learning algorithm jo ek nayi point ko classify karne ke liye uske **K sabse paas ke neighbors** dekhta hai
- **Instance-based learning** — KNN koi model nahi banata; woh poora training data yaad rakhta hai
- **Lazy learner** — training ke time kuch nahi seekhta, prediction ke time kaam karta hai
- **K** — ek hyperparameter; kitne neighbors dekhne hain yeh decide karta hai
- **Majority voting** — classification mein, jo class zyada neighbors mein hai woh predict hoti hai

### KNN kaise kaam karta hai? (Step by step)

1. **Training:** Poora dataset yaad kar lo (literally!)
2. **Naya point aaya:** Uski distance calculate karo har training point se
3. **K nearest neighbors dhundho:** Sabse kam distance wale K points chunno
4. **Vote karo:** Un K neighbors mein jo class zyada hai, woh predict karo

### Ek Seedha Example:

Socho tumhare paas ek nayi movie hai. Tum decide karna chahte ho ki yeh **Action** hai ya **Romance**.
- K=3 set karo
- Sabse paas ke 3 movies dhundho
- Agar 2 Action aur 1 Romance hai → **Action** predict karo!

In [ ]:
# Simple sample dataset — 2 features, 2 classes
# Imagine karo yeh students ke marks hain — Pass ya Fail
data = {
    'Maths_Score':   [30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85],
    'Science_Score': [25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80],
    'Result':        [0,   0,  0,  0,  0,  1,  1,  1,  1,  1,  1,  1]
    # 0 = Fail, 1 = Pass
}

df = pd.DataFrame(data)
print('Dataset:')
print(df)
print('\nClass distribution:')
print(df['Result'].value_counts())

In [ ]:
# Data visualise karte hain
colors = ['salmon' if r == 0 else 'steelblue' for r in df['Result']]

plt.figure(figsize=(7, 5))
plt.scatter(df['Maths_Score'], df['Science_Score'], c=colors, s=120, edgecolor='black', zorder=5)

# Ek nayi test point add karo
plt.scatter(52, 48, color='green', s=200, marker='*', zorder=10, label='Nayi Test Point ⭐')

plt.xlabel('Maths Score')
plt.ylabel('Science Score')
plt.title('KNN — Nayi Point ka Result kya hoga?')
plt.legend(['Pass (1)', 'Fail (0)', 'Nayi Test Point'])

# Manually legend theek karte hain
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', label='Pass (1)'),
    Patch(facecolor='salmon', label='Fail (0)'),
    plt.Line2D([0], [0], marker='*', color='w', markerfacecolor='green', markersize=15, label='Nayi Point')
]
plt.legend(handles=legend_elements)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 📌 Section 2 — Distance Functions

KNN mein **distance** sab kuch hai — jitna kam distance, utna zyada 'paas' hai neighbor.
Teen important distance functions hain:

**Yeh keywords yaad rakho:**
- **Euclidean Distance** — seedhi line ki doori (jaisi scale pe napte hain); sabse common
  $$d = \sqrt{\sum_{i=1}^{n}(x_i - y_i)^2}$$

- **Manhattan Distance** — grid mein chalke jaisi doori (jaise city blocks mein); outliers ke liye better
  $$d = \sum_{i=1}^{n}|x_i - y_i|$$

- **Minkowski Distance** — Euclidean aur Manhattan dono ka generalized form; p ek parameter hai
  $$d = \left(\sum_{i=1}^{n}|x_i - y_i|^p\right)^{1/p}$$
  - p = 1 → Manhattan Distance
  - p = 2 → Euclidean Distance
  - p = ∞ → Chebyshev Distance

**Sklearn mein default:** Minkowski with p=2 (yaani Euclidean)

In [ ]:
# Distance functions manually implement karte hain

def euclidean_distance(a, b):
    return np.sqrt(np.sum((np.array(a) - np.array(b))**2))

def manhattan_distance(a, b):
    return np.sum(np.abs(np.array(a) - np.array(b)))

def minkowski_distance(a, b, p):
    return np.sum(np.abs(np.array(a) - np.array(b))**p)**(1/p)

# Do points ke beech distance calculate karte hain
point_A = [2, 4]
point_B = [5, 8]

print(f'Point A: {point_A}')
print(f'Point B: {point_B}')
print()
print(f'Euclidean Distance  : {euclidean_distance(point_A, point_B):.4f}')
print(f'Manhattan Distance  : {manhattan_distance(point_A, point_B):.4f}')
print(f'Minkowski (p=1)     : {minkowski_distance(point_A, point_B, 1):.4f}  ← Manhattan jaisa')
print(f'Minkowski (p=2)     : {minkowski_distance(point_A, point_B, 2):.4f}  ← Euclidean jaisa')
print(f'Minkowski (p=3)     : {minkowski_distance(point_A, point_B, 3):.4f}')

In [ ]:
# Distance ko visually samajhte hain
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

A = np.array([1, 1])
B = np.array([4, 5])

# Euclidean
axes[0].scatter(*A, color='steelblue', s=150, zorder=5, label='Point A')
axes[0].scatter(*B, color='salmon', s=150, zorder=5, label='Point B')
axes[0].plot([A[0], B[0]], [A[1], B[1]], 'k-', lw=2, label=f'Euclidean = {euclidean_distance(A,B):.2f}')
axes[0].set_title('Euclidean Distance\n(Seedhi line ki doori)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 6)
axes[0].set_ylim(0, 7)

# Manhattan
axes[1].scatter(*A, color='steelblue', s=150, zorder=5, label='Point A')
axes[1].scatter(*B, color='salmon', s=150, zorder=5, label='Point B')
axes[1].plot([A[0], B[0]], [A[1], A[1]], 'g--', lw=2)
axes[1].plot([B[0], B[0]], [A[1], B[1]], 'g--', lw=2, label=f'Manhattan = {manhattan_distance(A,B):.2f}')
axes[1].set_title('Manhattan Distance\n(Grid mein chalke jaisi doori)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 6)
axes[1].set_ylim(0, 7)

plt.suptitle('Distance Functions ka Visual Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📌 Section 3 — KNN Kyun Use Karein?

### KNN ke Fayde (Pros)

| Faida | Explanation |
|---|---|
| Simple aur samajhne mein aasan | Koi complex math nahi — bas distance calculate karo |
| Non-parametric | Koi assumption nahi data distribution ke baare mein |
| Naye data ke saath instantly update | Training data add karo, model automatically update ho jaata hai |
| Classification aur Regression dono | Dono tasks ke liye use ho sakta hai |
| Complex decision boundaries | Non-linear patterns bhi capture kar sakta hai |

### KNN ki Kamiyaan (Cons)

| Kamiyaan | Explanation |
|---|---|
| **Slow prediction** | Har nayi point ke liye saare training data se distance calculate karna padta hai |
| **Large memory** | Poora training data memory mein rakhna padta hai |
| **Scaling zaroori** | Bina scaling ke bade features chote features ko dominate karte hain |
| **Curse of Dimensionality** | Zyada features hone pe distance meaningless ho jaati hai |
| **K ka sahi chunav zaroori** | Galat K → underfitting ya overfitting |
| **Imbalanced data pe kamzor** | Majority class dominate karti hai |

### K ki Value ka Effect

- **Chota K (jaise K=1)** → model bahut sensitive hai noise ke liye → **Overfitting**
- **Bada K (jaise K=100)** → model bahut smooth hai → **Underfitting**
- **Best K** → cross-validation ya elbow method se dhundho

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Thoda bada dataset banate hain
np.random.seed(42)
n = 200
X_demo = np.random.randn(n, 2)
# Class 1 agar sum > 0, warna class 0
y_demo = ((X_demo[:, 0] + X_demo[:, 1]) > 0).astype(int)

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_demo, y_demo, test_size=0.3, random_state=42)

# Alag alag K values try karte hain
k_values = [1, 3, 5, 7, 11, 15, 21, 31]
train_acc = []
test_acc  = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_d, y_train_d)
    train_acc.append(accuracy_score(y_train_d, knn.predict(X_train_d)))
    test_acc.append(accuracy_score(y_test_d, knn.predict(X_test_d)))

print('K ki Value  | Train Accuracy | Test Accuracy')
print('-' * 45)
for k, tr, te in zip(k_values, train_acc, test_acc):
    print(f'  K = {k:>3}    |    {tr:.4f}      |   {te:.4f}')

In [ ]:
# K vs Accuracy plot (Elbow Method)
plt.figure(figsize=(8, 4))
plt.plot(k_values, train_acc, 'o-', color='steelblue', label='Training Accuracy')
plt.plot(k_values, test_acc,  'o-', color='salmon',    label='Testing Accuracy')
plt.xlabel('K ki Value (Neighbors ki Count)')
plt.ylabel('Accuracy')
plt.title('K ki Value ka Effect — Overfitting vs Underfitting')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(k_values)
plt.tight_layout()
plt.show()
print('\nNote: Jahan train aur test accuracy ke beech gap kam ho aur test accuracy high ho — woh best K hai!')

---
## 📌 Section 4 — Mathematical Approach (Step by Step)

**KNN Algorithm ko ek example ke saath step by step samjhte hain:**

**Yeh keywords yaad rakho:**
- **Feature space** — ek aisa space jahan har dimension ek feature hai
- **Query point** — woh nayi point jisko classify karna hai
- **Sorted distances** — saari distances ko chote se bade order mein lagana
- **Majority class** — K neighbors mein jo class sabse zyada baar aaye

**Algorithm:**
1. Training data: $(x_1, y_1), (x_2, y_2), \ldots, (x_n, y_n)$
2. Query point: $x_q$
3. Har training point se distance: $d_i = dist(x_q, x_i)$
4. Distances sort karo, top K lo: $N_K(x_q) = \{x_{i_1}, x_{i_2}, \ldots, x_{i_K}\}$
5. Classification prediction:
$$\hat{y} = \text{argmax}_c \sum_{i \in N_K(x_q)} \mathbb{1}[y_i = c]$$
6. Regression prediction:
$$\hat{y} = \frac{1}{K} \sum_{i \in N_K(x_q)} y_i$$

In [ ]:
# KNN ko scratch se implement karte hain — samajhne ke liye

def knn_predict_scratch(X_train, y_train, query_point, K):
    """Ek nayi point ke liye KNN prediction — manually"""
    # Step 1: Har training point se Euclidean distance calculate karo
    distances = []
    for i, x in enumerate(X_train):
        dist = np.sqrt(np.sum((np.array(x) - np.array(query_point))**2))
        distances.append((dist, y_train[i]))

    # Step 2: Distance ke hisaab se sort karo
    distances.sort(key=lambda x: x[0])

    # Step 3: Top K neighbors lo
    k_nearest = distances[:K]
    print(f'Top {K} Nearest Neighbors (distance, class):')
    for dist, cls in k_nearest:
        print(f'  Distance = {dist:.3f}, Class = {cls} ({"Pass" if cls==1 else "Fail"})')

    # Step 4: Majority vote karo
    labels = [label for _, label in k_nearest]
    prediction = max(set(labels), key=labels.count)
    print(f'\nVoting result: {labels}')
    print(f'Predicted class: {prediction} ({"Pass" if prediction==1 else "Fail"})')
    return prediction

# Sample data use karte hain
X_train_sample = df[['Maths_Score', 'Science_Score']].values
y_train_sample = df['Result'].values
nayi_student   = [52, 48]  # Nayi student ke marks

print('=' * 50)
print(f'Nayi Student ke marks: Maths={nayi_student[0]}, Science={nayi_student[1]}')
print('=' * 50)
pred = knn_predict_scratch(X_train_sample, y_train_sample, nayi_student, K=3)

---
## 📌 Section 5 — EDA aur Dataset

**Yeh keywords yaad rakho:**
- **EDA** — model banane se pehle data explore karna — patterns, distributions, missing values dhundhna
- **pairplot** — har pair of features ke beech scatter plot dikhata hai; patterns samajhne mein helpful
- **heatmap** — features ke beech correlation dikhata hai

Hum ab ek **larger dataset** use karenge — fruit classification:
- Features: Weight (grams), Sweetness (1-10), Diameter (cm)
- Classes: Apple (0), Orange (1), Mango (2)

In [ ]:
# Fruit dataset banate hain
np.random.seed(42)
n_each = 60

# Apple
apple_weight    = np.random.normal(182, 15, n_each)
apple_sweet     = np.random.normal(5.5,  1, n_each)
apple_diameter  = np.random.normal(7.5, 0.5, n_each)

# Orange
orange_weight   = np.random.normal(130, 12, n_each)
orange_sweet    = np.random.normal(6.5,  1, n_each)
orange_diameter = np.random.normal(8.5, 0.5, n_each)

# Mango
mango_weight    = np.random.normal(250, 20, n_each)
mango_sweet     = np.random.normal(8.5,  1, n_each)
mango_diameter  = np.random.normal(10.5, 0.8, n_each)

fruit_df = pd.DataFrame({
    'Weight_g':     np.concatenate([apple_weight,   orange_weight,   mango_weight]),
    'Sweetness':    np.concatenate([apple_sweet,    orange_sweet,    mango_sweet]),
    'Diameter_cm':  np.concatenate([apple_diameter, orange_diameter, mango_diameter]),
    'Fruit':        ['Apple']*n_each + ['Orange']*n_each + ['Mango']*n_each
})
fruit_df = fruit_df.round(2)

print('Fruit Dataset:')
print(fruit_df.shape)
print()
print(fruit_df.head(8))
print()
print('Class distribution:')
print(fruit_df['Fruit'].value_counts())

In [ ]:
# EDA Charts
print('Basic Statistics:')
print(fruit_df.describe())
print()
print('Missing values:')
print(fruit_df.isnull().sum())

In [ ]:
# Pairplot — features ke beech relationship
sns.pairplot(fruit_df, hue='Fruit', palette={'Apple': 'steelblue', 'Orange': 'darkorange', 'Mango': 'gold'})
plt.suptitle('Fruit Dataset — Pairplot (Har feature pair ka scatter)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots — har feature ka distribution by class
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
features_to_plot = ['Weight_g', 'Sweetness', 'Diameter_cm']
colors = ['steelblue', 'darkorange', 'gold']

for ax, feat in zip(axes, features_to_plot):
    fruit_df.boxplot(column=feat, by='Fruit', ax=ax)
    ax.set_title(f'{feat} — Fruit ke Hisaab se')
    ax.set_xlabel('Fruit')

plt.suptitle('Har Feature ka Distribution — Fruit ke Hisaab se')
plt.tight_layout()
plt.show()

---
## 📌 Section 6 — Preprocessing — Scaling Kyun Zaroori Hai?

**Yeh keywords yaad rakho:**
- **Feature Scaling** — saari features ko same scale pe laana
- **StandardScaler** — mean=0, std=1 karta hai
- **MinMaxScaler** — 0 se 1 ke beech scale karta hai

### KNN mein Scaling BAHUT zaroori kyun hai?

Socho ek dataset mein:
- Weight: 100g se 300g (range = 200)
- Sweetness: 1 se 10 (range = 9)

Bina scaling ke, Weight wali feature distance mein dominate karegi kyunki uski values zyada badi hain.
Sweetness ka contribution negligible ho jaayega — chahe woh actually zyada important ho!

**Scaling ke baad:** Dono features equally contribute karte hain distance calculation mein.

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# Features aur target
X = fruit_df[['Weight_g', 'Sweetness', 'Diameter_cm']]
y = fruit_df['Fruit']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling se pehle aur baad ka comparison
print('SCALING SE PEHLE — Features ki range:')
print(X_train.describe().loc[['min', 'max', 'mean', 'std']])

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('\nSCALING KE BAAD — Features ki range:')
print(pd.DataFrame(X_train_sc, columns=X.columns).describe().loc[['min', 'max', 'mean', 'std']].round(3))

In [ ]:
# Scaling ka effect on distance — ek example
point1 = X_train.iloc[0].values
point2 = X_train.iloc[1].values

point1_sc = X_train_sc[0]
point2_sc = X_train_sc[1]

dist_before = np.sqrt(np.sum((point1 - point2)**2))
dist_after  = np.sqrt(np.sum((point1_sc - point2_sc)**2))

print('Scaling se pehle distance:', round(dist_before, 4), '← Weight feature dominate kar rahi hai')
print('Scaling ke baad  distance:', round(dist_after, 4),  '← Ab properly balanced hai')

---
## 📌 Section 7 — Model Training — Best K Kaise Chunein

**Yeh keywords yaad rakho:**
- **Hyperparameter** — woh parameter jo model training se pehle set karna padta hai (jaise K)
- **Cross-validation** — data ko multiple times split karo, har split pe train aur test karo — reliable estimate ke liye
- **Elbow Method** — K vs Error rate ka graph banao, jahan curve mein bend aaye woh K best hai
- **n_neighbors** — sklearn mein K ka naam

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Alag alag K values try karo
k_range = range(1, 31)
train_scores = []
test_scores  = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, metric='minkowski', p=2)  # p=2 yaani Euclidean
    knn.fit(X_train_sc, y_train)
    train_scores.append(accuracy_score(y_train, knn.predict(X_train_sc)))
    test_scores.append(accuracy_score(y_test, knn.predict(X_test_sc)))

best_k = k_range[test_scores.index(max(test_scores))]
print(f'Best K: {best_k} (Test Accuracy = {max(test_scores):.4f})')

In [ ]:
# Elbow curve plot
plt.figure(figsize=(10, 4))
plt.plot(list(k_range), train_scores, 'o-', color='steelblue', label='Training Accuracy')
plt.plot(list(k_range), test_scores,  'o-', color='salmon',    label='Testing Accuracy')
plt.axvline(best_k, color='green', linestyle='--', label=f'Best K = {best_k}')
plt.xlabel('K ki Value')
plt.ylabel('Accuracy')
plt.title('Elbow Method — Best K dhundho')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(range(1, 31, 2))
plt.tight_layout()
plt.show()

In [ ]:
# Best K se final model train karo
final_knn = KNeighborsClassifier(
    n_neighbors=best_k,
    metric='minkowski',  # Distance function
    p=2,                 # p=2 yaani Euclidean distance
    weights='uniform'    # Har neighbor ka equal vote
)
final_knn.fit(X_train_sc, y_train)
y_pred = final_knn.predict(X_test_sc)

print(f'Final Model — K={best_k}')
print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}')

---
## 📌 Section 8 — Confusion Matrix aur Accuracy

**Yaad karo:**
- **Confusion Matrix** — actual vs predicted values ka breakdown
- **TP / TN / FP / FN** — correct aur incorrect predictions ke types
- **ConfusionMatrixDisplay** — modern sklearn way (recommended)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

cm = confusion_matrix(y_test, y_pred, labels=['Apple', 'Orange', 'Mango'])
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('\nConfusion Matrix (raw):')
print(cm)

In [ ]:
# TARIKA 1 — Purana: seaborn heatmap
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Apple', 'Orange', 'Mango'],
            yticklabels=['Apple', 'Orange', 'Mango'])
plt.xlabel('Predicted (Model ne kya bola)')
plt.ylabel('Actual (Actually kya tha)')
plt.title('Confusion Matrix — seaborn heatmap (purana tarika)')
plt.tight_layout()
plt.show()

In [ ]:
# TARIKA 2 — Modern: ConfusionMatrixDisplay (recommended)
ConfusionMatrixDisplay.from_estimator(
    final_knn, X_test_sc, y_test,
    display_labels=['Apple', 'Orange', 'Mango'],
    cmap='Blues'
)
plt.title('Confusion Matrix — ConfusionMatrixDisplay (naya tarika)')
plt.tight_layout()
plt.show()

---
## 📌 Section 9 — Precision, Recall, F1 Score

**Yaad karo:**
- **Precision** = TP / (TP + FP) — predict kiye positives mein se kitne sahi the
- **Recall** = TP / (TP + FN) — actual positives mein se kitne pakde
- **F1 Score** = Precision aur Recall ka harmonic mean

In [ ]:
from sklearn.metrics import classification_report

print('Pura Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Apple', 'Orange', 'Mango']))

---
## 📌 Section 10 — KNN Regression (Bonus)

**Yeh keywords yaad rakho:**
- **KNN Regression** — K nearest neighbors ki values ka **average** predict karta hai (number predict karna)
- **KNeighborsRegressor** — sklearn mein KNN regression ke liye class
- **MAE (Mean Absolute Error)** — prediction error ka average (actual vs predicted mein average farq)

$$\hat{y} = \frac{1}{K} \sum_{i \in N_K(x_q)} y_i$$

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Simple regression dataset — Ghar ki kimat predict karna
np.random.seed(0)
area = np.sort(np.random.uniform(500, 3000, 100))
price = 50 * area + np.random.normal(0, 20000, 100) + 100000

X_reg = area.reshape(-1, 1)
y_reg = price

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

sc_reg = StandardScaler()
X_tr_sc = sc_reg.fit_transform(X_tr)
X_te_sc = sc_reg.transform(X_te)

knn_reg = KNeighborsRegressor(n_neighbors=5)
knn_reg.fit(X_tr_sc, y_tr)
y_pred_reg = knn_reg.predict(X_te_sc)

print(f'MAE (Mean Absolute Error): ₹{mean_absolute_error(y_te, y_pred_reg):,.0f}')
print(f'R² Score (1 = perfect)   : {r2_score(y_te, y_pred_reg):.4f}')

# Plot
plt.figure(figsize=(8, 4))
plt.scatter(X_tr, y_tr, alpha=0.5, label='Training data', color='steelblue')
plt.scatter(X_te, y_te, alpha=0.7, label='Actual test', color='salmon')
x_line = np.linspace(500, 3000, 300).reshape(-1, 1)
plt.plot(x_line, knn_reg.predict(sc_reg.transform(x_line)), color='green', lw=2, label='KNN Prediction')
plt.xlabel('Area (sq ft)')
plt.ylabel('Ghar ki Kimat (₹)')
plt.title('KNN Regression — Ghar ki Kimat Predict Karna')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## ✅ Summary — Kya Kya Sikha

| Topic | Key Takeaway |
|---|---|
| KNN Algorithm | Neighbors ki majority class predict karta hai |
| Distance Functions | Euclidean, Manhattan, Minkowski |
| K ki Value | Chota K = overfit, Bada K = underfit; elbow method se best K dhundho |
| Scaling ZAROORI | Bina scaling ke distance meaningless ho jaati hai |
| Lazy Learner | Training mein kuch nahi seekhta, prediction mein kaam karta hai |
| Confusion Matrix | ConfusionMatrixDisplay use karo (modern tarika) |
| KNN Regression | Neighbors ka average predict karta hai |

---
*Aage: Yeh sab real Kaggle dataset pe apply karo → Notebook 2 dekho*